# Graph Traversal & Measures with NetworkX (Game of Thrones)
Run the cells in order. Export the GEXF file at the end and open it in Gephi.

## Step 1: Install and import

In [ ]:
%pip install -q networkx pandas matplotlib
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import time

## Step 2: Load the CSV (edges file, e.g. stormofswords.csv)

In [ ]:
fname = "stormofswords.csv"
df = pd.read_csv(fname)
df.columns = [c.strip().lower() for c in df.columns]
print(fname)
df.head()

## Step 3: Build the graph

In [ ]:
G = nx.from_pandas_edgelist(df, source="source", target="target",
                            edge_attr="weight" if "weight" in df.columns else None)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Connected:", nx.is_connected(G))
print("Sample nodes:", list(G.nodes())[:20])

## Step 4: Choose start and target characters

In [ ]:
def find(name):
    return [n for n in G.nodes() if name.lower() in str(n).lower()]

jon_matches = find("jon")
cersei_matches = find("cersei")
print(jon_matches, cersei_matches)

start = jon_matches[0] if jon_matches else "Jon-Snow"
target = cersei_matches[0] if cersei_matches else "Cersei-Lannister"
assert start in G and target in G, "Update start/target to match the node names printed above"

## Step 5: Graph traversal (BFS, DFS, shortest paths)

In [ ]:
# BFS
t0 = time.perf_counter()
bfs_order = list(nx.bfs_tree(G, source=start).nodes())
bfs_time = time.perf_counter() - t0

# DFS
t0 = time.perf_counter()
dfs_order = list(nx.dfs_preorder_nodes(G, source=start))
dfs_time = time.perf_counter() - t0

print("BFS first 15:", bfs_order[:15])
print("DFS first 15:", dfs_order[:15])
print(f"BFS time: {bfs_time:.6f}s | DFS time: {dfs_time:.6f}s")

# BFS layers = degrees of separation
layers = dict(enumerate(nx.bfs_layers(G, start)))
for depth, nodes in layers.items():
    print(f"Depth {depth}: {len(nodes)} nodes")

# Shortest paths
path = nx.shortest_path(G, start, target)
print("Shortest path:", path, "| length:", len(path) - 1)

for u, v, d in G.edges(data=True):
    d["distance"] = 1 / d.get("weight", 1)
wpath = nx.dijkstra_path(G, start, target, weight="distance")
print("Weighted (Dijkstra) path:", wpath)

## Step 6: Graph measures

In [ ]:
GC = G.subgraph(max(nx.connected_components(G), key=len))

print("Density:", nx.density(G))
print("Diameter:", nx.diameter(GC))
print("Avg path length:", nx.average_shortest_path_length(GC))
print("Avg clustering:", nx.average_clustering(G))

deg = nx.degree_centrality(G)
btw = nx.betweenness_centrality(G)
clo = nx.closeness_centrality(G)

metrics = pd.DataFrame({"degree": deg, "betweenness": btw, "closeness": clo})
metrics.sort_values("betweenness", ascending=False).head(10)

## Step 7: Community detection

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities, modularity
comms = greedy_modularity_communities(G)
print("Communities:", len(comms), "| Modularity:", modularity(G, comms))

## Step 8: Quick NetworkX plot (to compare with Gephi later)

In [ ]:
node_comm = {n: i for i, c in enumerate(comms) for n in c}
plt.figure(figsize=(12, 9))
pos = nx.spring_layout(G, seed=42, k=0.3)
nx.draw_networkx_nodes(G, pos, node_size=[3000 * btw[n] + 30 for n in G],
                       node_color=[node_comm[n] for n in G], cmap="tab10")
nx.draw_networkx_edges(G, pos, alpha=0.2)
nx.draw_networkx_labels(G, pos, labels={n: n for n in G if btw[n] > 0.03}, font_size=8)
plt.axis("off")
plt.title("NetworkX: size = betweenness, color = community")
plt.savefig("networkx_plot.png", dpi=200, bbox_inches="tight")
plt.show()

## Step 9: Export for Gephi

In [ ]:
nx.set_node_attributes(G, deg, "nx_degree")
nx.set_node_attributes(G, btw, "nx_betweenness")
nx.set_node_attributes(G, clo, "nx_closeness")
nx.set_node_attributes(G, node_comm, "nx_community")
for depth, nodes in layers.items():
    for n in nodes:
        G.nodes[n]["bfs_depth"] = depth

nx.write_gexf(G, "got_network.gexf")
metrics.to_csv("networkx_metrics.csv")
print("Saved got_network.gexf, networkx_metrics.csv, and networkx_plot.png locally.")